# ACC102 Track 4 Notebook  
## WRDS Stock Performance Analyser

This notebook reorganises the original `Final.py` Streamlit application into a clear analytical workflow suitable for ACC102 submission.

**Track:** Track 4 – Interactive Data Analysis Tool  
**Project focus:** Compare the performance and risk of two stocks against a market benchmark using WRDS CRSP data.

---

## 1. Problem Definition and Intended User

### Analytical problem
This project addresses a practical finance question: **how can a user compare the return and risk performance of two stocks relative to a market benchmark over a selected period?**

### Intended user
The tool is designed for:
- beginner investors,
- finance and accounting students,
- users who need a simple interactive tool for comparing stock performance.

### Why this matters
A user-facing tool should do more than display prices. It should transform raw financial data into interpretable outputs such as:
- adjusted price trends,
- cumulative returns,
- annualised return and volatility,
- Sharpe ratio,
- downloadable comparison tables.

This notebook demonstrates the Python analysis that supports the final interactive product.

## 2. Data Source and Variable Overview

### Data source
The project uses **WRDS CRSP** data.

### Why this dataset was selected
CRSP is widely used in academic and professional finance research because it provides structured historical stock data suitable for return and risk analysis.

### Main variables used
- `permno`: permanent security identifier in CRSP
- `date`: trading date
- `prc`: stock price
- `cfacpr`: price adjustment factor

### Data access note
To run this notebook fully, the user must have valid WRDS credentials.

## 3. Package Import and Environment Setup

This section imports the packages used in the original Streamlit project.  
The original `.py` file included automatic package installation. In notebook form, the analytical logic is kept, but installation is not forced inside the workflow.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import wrds
from datetime import date, timedelta

## 4. Core Functions

The following functions come from the original code and are reorganised here as reusable analytical components.

### Function roles
- `connect_wrds`: connect to WRDS
- `get_permno_by_ticker`: map ticker to CRSP permno
- `get_price_data`: retrieve and prepare adjusted price data
- `compute_metrics`: compute return and risk indicators

In [ ]:
def connect_wrds(username: str, password: str):
    """Create a WRDS connection using username and password."""
    try:
        return wrds.Connection(wrds_username=username, wrds_password=password)
    except Exception as e:
        print(f"WRDS Connection Failed: {e}")
        return None


def get_permno_by_ticker(db, ticker: str):
    """Get PERMNO from ticker using CRSP stocknames."""
    ticker = ticker.upper().strip()
    query = f"""
    SELECT permno
    FROM crsp.stocknames
    WHERE ticker = '{ticker}'
    ORDER BY namedt DESC
    LIMIT 1
    """
    result = db.raw_sql(query)
    if result.empty:
        print(f"No PERMNO found for {ticker}. Please check the ticker symbol.")
        return None
    return int(result["permno"].iloc[0])


def get_price_data(permno: int, start: str, end: str, username: str):
    """Query WRDS CRSP daily stock file and compute derived variables."""
    db = wrds.Connection(wrds_username=username)
    query = f"""
    SELECT date, prc, cfacpr
    FROM crsp.dsf
    WHERE permno = {permno}
      AND date >= '{start}'
      AND date <= '{end}'
    ORDER BY date
    """
    df = db.raw_sql(query)
    db.close()

    if df.empty:
        return pd.DataFrame()

    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")

    # Adjusted price
    df["adj_prc"] = df["prc"] * df["cfacpr"]

    # Log return and cumulative return
    df["daily_return"] = np.log(df["adj_prc"] / df["adj_prc"].shift(1))
    df["cum_return"] = (1 + df["daily_return"]).cumprod() - 1

    return df


def compute_metrics(price_series: pd.Series, return_series: pd.Series):
    """Compute academic-standard performance metrics."""
    p = price_series.dropna()
    r = return_series.dropna()

    if len(p) < 2:
        return {
            "Total Return (%)": 0,
            "Annual Return (%)": 0,
            "Annual Volatility (%)": 0,
            "Sharpe Ratio": 0,
            "Final Price ($)": 0
        }

    total_ret = (p.iloc[-1] / p.iloc[0] - 1) * 100
    ann_ret = ((1 + total_ret / 100) ** (252 / len(p)) - 1) * 100
    ann_vol = r.std() * np.sqrt(252) * 100
    sharpe = (r.mean() * 252) / (r.std() * np.sqrt(252)) if r.std() != 0 else np.nan

    return {
        "Total Return (%)": round(total_ret, 2),
        "Annual Return (%)": round(ann_ret, 2),
        "Annual Volatility (%)": round(ann_vol, 2),
        "Sharpe Ratio": round(sharpe, 3),
        "Final Price ($)": round(p.iloc[-1], 2)
    }

## 5. User Inputs

The original Streamlit app collects:
- WRDS username and password,
- two stock tickers,
- one benchmark ticker,
- a date range.

In notebook form, these are declared as ordinary variables. Replace them with your own values when running the notebook.

In [ ]:
# Replace these placeholders with valid WRDS credentials when running the notebook
wrds_user = "your_wrds_username"
wrds_password = "your_wrds_password"

# Example tickers from the original app
ticker1 = "AAPL"
ticker2 = "MSFT"
benchmark_ticker = "SPY"

# Default period: last 3 years
end_date = date.today()
start_date = end_date - timedelta(days=365 * 3)

start_str = str(start_date)
end_str = str(end_date)

print("Selected tickers:", ticker1, ticker2, benchmark_ticker)
print("Date range:", start_str, "to", end_str)

## 6. Connect to WRDS and Retrieve Security Identifiers

This step converts stock tickers into CRSP `permno` identifiers.  
The use of `permno` is important because it provides a stable identifier for querying the daily stock file.

In [ ]:
# Run this cell only when you have valid WRDS credentials
# db_conn = connect_wrds(wrds_user, wrds_password)

# permno1 = get_permno_by_ticker(db_conn, ticker1)
# permno2 = get_permno_by_ticker(db_conn, ticker2)
# permno_bench = get_permno_by_ticker(db_conn, benchmark_ticker)

# print(ticker1, permno1)
# print(ticker2, permno2)
# print(benchmark_ticker, permno_bench)

## 7. Data Loading and Preparation

After obtaining `permno` values, the notebook queries WRDS CRSP daily stock data and prepares:
- adjusted price,
- log daily return,
- cumulative return.

This is the core data transformation stage of the project.

In [ ]:
# Run this cell after obtaining valid PERMNO values
# df1 = get_price_data(permno1, start_str, end_str, wrds_user)
# df2 = get_price_data(permno2, start_str, end_str, wrds_user)
# df_bench = get_price_data(permno_bench, start_str, end_str, wrds_user)

# print(df1.head())
# print(df2.head())
# print(df_bench.head())

## 8. Initial Data Quality Check

Before analysis, the project checks whether the returned datasets are empty.  
This prevents later charting or metric errors and improves workflow reliability.

In [ ]:
# Example validation step
# if df1.empty or df2.empty or df_bench.empty:
#     raise ValueError("No data returned for one or more selected stocks. Please check tickers or date range.")

## 9. Compute Performance Metrics

The notebook now calculates the main summary indicators used in the final tool:
- total return,
- annual return,
- annual volatility,
- Sharpe ratio,
- final adjusted price.

In [ ]:
# metrics1 = compute_metrics(df1["adj_prc"], df1["daily_return"])
# metrics2 = compute_metrics(df2["adj_prc"], df2["daily_return"])
# metrics_bench = compute_metrics(df_bench["adj_prc"], df_bench["daily_return"])

# metrics_df = pd.DataFrame({
#     ticker1: metrics1,
#     ticker2: metrics2,
#     benchmark_ticker + " (Market)": metrics_bench
# }).T

# metrics_df

## 10. Visual Analysis

The original application presents four key visual outputs:
1. adjusted price comparison,
2. normalised price comparison,
3. cumulative return comparison,
4. risk-return scatter plot.

These charts help convert raw financial data into user-friendly insights.

In [ ]:
# Example plotting code from the original project
# Execute after loading df1, df2, and df_bench

# 1. Absolute price comparison
# fig_price = go.Figure()
# fig_price.add_trace(go.Scatter(x=df1.index, y=df1["adj_prc"], name=ticker1))
# fig_price.add_trace(go.Scatter(x=df2.index, y=df2["adj_prc"], name=ticker2))
# fig_price.add_trace(go.Scatter(x=df_bench.index, y=df_bench["adj_prc"], name=benchmark_ticker))
# fig_price.update_layout(title="Absolute Price Comparison", xaxis_title="Date", yaxis_title="Adjusted Price ($)")
# fig_price.show()

# 2. Normalised price comparison
# df1["norm_prc"] = df1["adj_prc"] / df1["adj_prc"].iloc[0] * 100
# df2["norm_prc"] = df2["adj_prc"] / df2["adj_prc"].iloc[0] * 100
# df_bench["norm_prc"] = df_bench["adj_prc"] / df_bench["adj_prc"].iloc[0] * 100
# fig_norm = go.Figure()
# fig_norm.add_trace(go.Scatter(x=df1.index, y=df1["norm_prc"], name=ticker1))
# fig_norm.add_trace(go.Scatter(x=df2.index, y=df2["norm_prc"], name=ticker2))
# fig_norm.add_trace(go.Scatter(x=df_bench.index, y=df_bench["norm_prc"], name=benchmark_ticker))
# fig_norm.update_layout(title="Normalised Price Comparison (Base = 100)")
# fig_norm.show()

# 3. Cumulative return comparison
# fig_cum = go.Figure()
# fig_cum.add_trace(go.Scatter(x=df1.index, y=df1["cum_return"] * 100, name=ticker1))
# fig_cum.add_trace(go.Scatter(x=df2.index, y=df2["cum_return"] * 100, name=ticker2))
# fig_cum.add_trace(go.Scatter(x=df_bench.index, y=df_bench["cum_return"] * 100, name=benchmark_ticker))
# fig_cum.update_layout(title="Cumulative Return (%)")
# fig_cum.show()

# 4. Risk-return scatter plot
# scatter_data = pd.DataFrame({
#     "Stock": [ticker1, ticker2, benchmark_ticker],
#     "Annual Return (%)": [
#         metrics1["Annual Return (%)"],
#         metrics2["Annual Return (%)"],
#         metrics_bench["Annual Return (%)"]
#     ],
#     "Annual Volatility (%)": [
#         metrics1["Annual Volatility (%)"],
#         metrics2["Annual Volatility (%)"],
#         metrics_bench["Annual Volatility (%)"]
#     ]
# })
# fig_scatter = px.scatter(
#     scatter_data,
#     x="Annual Volatility (%)",
#     y="Annual Return (%)",
#     color="Stock",
#     text="Stock",
#     title="Risk (Volatility) vs Return"
# )
# fig_scatter.update_traces(textposition="top center")
# fig_scatter.show()

## 11. Academic Interpretation Framework

The Streamlit app includes a short interpretation section that turns results into user-facing findings.  
After running the notebook with real data, this section should be updated with the actual results observed in the selected period.

### Suggested interpretation dimensions
- Which stock delivered the highest annual return?
- Which stock had the highest volatility?
- Which stock showed the strongest risk-adjusted performance?
- How did both stocks perform relative to the benchmark?

In [ ]:
# Example structure for interpretation after real execution
# print("Key Findings:")
# print(f"1. Annual return comparison: {ticker1}, {ticker2}, and {benchmark_ticker}.")
# print(f"2. Volatility comparison based on annualised volatility metrics.")
# print(f"3. Risk-adjusted comparison using Sharpe ratio.")

## 12. Raw Comparison Table and Export Preparation

The original app also prepares a merged data table for display and CSV export.  
This supports transparency and allows users to download the analysis output.

In [ ]:
# raw_data = pd.DataFrame({
#     f"{ticker1}_Adj_Price": df1["adj_prc"],
#     f"{ticker2}_Adj_Price": df2["adj_prc"],
#     f"{benchmark_ticker}_Adj_Price": df_bench["adj_prc"],
#     f"{ticker1}_Daily_Return": df1["daily_return"],
#     f"{ticker2}_Daily_Return": df2["daily_return"],
#     f"{benchmark_ticker}_Daily_Return": df_bench["daily_return"]
# }).round(4)

# raw_data.head()

## 13. Link to the Final Interactive Product

This notebook shows the Python analytical workflow.  
The final Track 4 product is the separate **Streamlit application**, where the same logic is presented through an interactive interface.

### Relationship between notebook and app
- **Notebook:** demonstrates analysis workflow clearly for marking
- **Streamlit app:** delivers user-facing interactivity and product value

## 14. Limitations

This project has several limitations:

1. **WRDS dependency**  
   The workflow depends on valid WRDS access credentials.

2. **Restricted variable scope**  
   The current version uses a limited set of CRSP variables (`prc`, `cfacpr`) and does not include broader firm or market variables.

3. **Simple performance metrics**  
   The project focuses on descriptive return-risk metrics and does not use more advanced portfolio or econometric models.

4. **User-selected period sensitivity**  
   Results may vary substantially depending on the chosen sample period.

## 15. Conclusion

This notebook reorganises the original code into a structured analytical narrative:

**problem → data source → retrieval → cleaning/preparation → metric computation → visual analysis → interpretation → product output**

That structure is important for ACC102 because it demonstrates that the final interactive tool is supported by substantive Python analysis rather than interface presentation alone.